In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
train = pd.read_csv("dataset.csv")
test  = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Ankit


In [ ]:
# Check for nulls
print("\nMissing values in train:")
print(train.isnull().sum())

print("\nMissing values in test:")
print(test.isnull().sum())

In [ ]:
# Class balance — how many 0s vs 1s?
print("\nResponse distribution:")
print(train["Response"].value_counts())
print(train["Response"].value_counts(normalize=True).round(3))

In [ ]:
# Quick look at categorical columns
for col in ["Gender", "Vehicle_Age", "Vehicle_Damage"]:
    print(f"\n{col}:")
    print(train[col].value_counts())

In [ ]:
# Always apply the same encoding to both train and test together

gender_map = {"Male": 1, "Female": 0}
train["Gender"] = train["Gender"].map(gender_map)
test["Gender"]  = test["Gender"].map(gender_map)

# --- Vehicle_Damage: Yes=1, No=0 ---
damage_map = {"Yes": 1, "No": 0}
train["Vehicle_Damage"] = train["Vehicle_Damage"].map(damage_map)
test["Vehicle_Damage"]  = test["Vehicle_Damage"].map(damage_map)

# --- Vehicle_Age: Ordinal encoding (order matters) ---
age_map = {"< 1 Year": 0, "1-2 Year": 1, "> 2 Years": 2}
train["Vehicle_Age"] = train["Vehicle_Age"].map(age_map)
test["Vehicle_Age"]  = test["Vehicle_Age"].map(age_map)

In [ ]:
# Verify — first 5 rows should now be all numeric
print(train.head())
print("\nDtypes:\n", train.dtypes)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Columns to scale
scale_cols = ["Age", "Annual_Premium", "Vintage", "Region_Code", "Policy_Sales_Channel"]

# Separate features and target
X = train.drop(columns=["id", "Response"])
y = train["Response"]
X_test = test.drop(columns=["id"])

print("Features shape:", X.shape)
print("Test shape    :", X_test.shape)

In [ ]:
scaler = StandardScaler()

# Fit ONLY on train, then transform both
# Never fit on test — that would cause data leakage
X[scale_cols]      = scaler.fit_transform(X[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
scaler = StandardScaler()

# Fit ONLY on train, then transform both
# Never fit on test — that would cause data leakage
X[scale_cols]      = scaler.fit_transform(X[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
# Verify scaling worked — mean should be ~0, std ~1
print(X[scale_cols].describe().round(2))

In [ ]:
# Final check — all columns numeric, no NaNs introduced
print("\nAny nulls after scaling?", X.isnull().sum().sum())
print("\nSample of processed features:")
print(X.head())

In [ ]:
print("Train features:", X.columns.tolist())
print("Test features :", X_test.columns.tolist())
print("Columns match :", X.columns.tolist() == X_test.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% validation
    random_state=42,     # for reproducibility
    stratify=y           # preserve class balance in both splits
)

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)
print("\nClass balance in y_train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nClass balance in y_val:")
print(y_val.value_counts(normalize=True).round(3))

In [ ]:
# scale_pos_weight = count of majority class / count of minority class
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f"Negative (0): {neg}")
print(f"Positive (1): {pos}")
print(f"scale_pos_weight: {scale:.2f}")

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale,   # handles class imbalance
    random_state=42,
    eval_metric="auc",
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],   # monitors val performance each round
    verbose=20                    # prints every 20 rounds
)
print("\nXGBoost training complete!")

In [ ]:
# Sanity check
train_score = xgb_model.score(X_train, y_train)
val_score   = xgb_model.score(X_val, y_val)

print(f"Train Accuracy : {train_score:.4f}")
print(f"Val Accuracy   : {val_score:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)

# Get predictions
y_pred      = xgb_model.predict(X_val)
y_pred_prob = xgb_model.predict_proba(X_val)[:, 1]  # probability of class 1

# Core metrics
print(f"Accuracy : {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_pred_prob):.4f}")

In [ ]:
# Precision, Recall, F1 per class
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Not Interested", "Interested"]))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)

print("\nConfusion Matrix:")
print(f"                  Predicted 0    Predicted 1")
print(f"Actual 0 (No)  :    {cm[0][0]:6d}        {cm[0][1]:6d}")
print(f"Actual 1 (Yes) :    {cm[1][0]:6d}        {cm[1][1]:6d}")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    "Feature"   : X_train.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importances:")
print(importance_df.to_string(index=False))

In [ ]:
# Generate predictions on test set
test_pred      = xgb_model.predict(X_test)
test_pred_prob = xgb_model.predict_proba(X_test)[:, 1]  # probability of class 1

In [ ]:
# Quick sanity check before saving
print("Test predictions shape :", test_pred.shape)
print("Sample predictions     :", test_pred[:10])
print("Sample probabilities   :", test_pred_prob[:10].round(3))
print("\nPrediction distribution:")
print(pd.Series(test_pred).value_counts())

In [ ]:
# Build submission dataframe
submission = pd.DataFrame({
    "id"      : test["id"],         # original id from test.csv
    "Response": test_pred           # 0 or 1
})

print("\nSubmission sample:")
print(submission.head(10))
print("\nSubmission shape:", submission.shape)

In [ ]:
# Save to CSV
submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully!")

In [ ]:
submission_prob = pd.DataFrame({
    "id"      : test["id"],
    "Response": test_pred_prob.round(4)   # probability of being interested
})

submission_prob.to_csv("submission_prob.csv", index=False)
print("submission_prob.csv saved successfully!")

In [ ]:
import os

# Check if files were created
files_to_check = ["submission.csv", "submission_prob.csv"]

for f in files_to_check:
    if os.path.exists(f):
        print(f"✅ {f} exists — size: {os.path.getsize(f)} bytes")
    else:
        print(f"❌ {f} NOT found — run the saving code again")

In [ ]:
import joblib
from xgboost import XGBClassifier

# Save the model
joblib.dump(xgb_model, "xgb_model.pkl")
print("✅ Model saved as xgb_model.pkl")

# Save the scaler too — you'll need it to process new data later
joblib.dump(scaler, "scaler.pkl")
print("✅ Scaler saved as scaler.pkl")

In [ ]:
# Download both
from google.colab import files

files.download("xgb_model.pkl")
files.download("scaler.pkl")
files.download("submission.csv")